In [1]:
# Importar librerías
import pandas as pd
import os
from src.ingestion import *

In [2]:
# Generar lista de tickers del fichero constituents del S&P 500
data_folder = "data"
os.makedirs(data_folder, exist_ok=True) # Crear carpeta si no existe
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

502

In [3]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector","GICS Sub-Industry", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "SubIndustry", 
    "Date added": "DateAdded"
    }, inplace=True)

# Eliminar espacios en los nombres de los sectores y sub-industrias
df_tickers["Sector"] = df_tickers["Sector"].str.replace(" ", "")
df_tickers["SubIndustry"] = df_tickers["SubIndustry"].str.replace(" ", "")

df_tickers.head()

,Ticker,Sector,SubIndustry,DateAdded
0,MMM,Industrials,IndustrialConglomerates,1957-03-04
1,AOS,Industrials,BuildingProducts,2017-07-26
2,ABT,HealthCare,HealthCareEquipment,1957-03-04
3,ABBV,HealthCare,Biotechnology,2012-12-31
4,ACN,InformationTechnology,ITConsulting&OtherServices,2011-07-06


In [4]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)
df_index = extraer_precios(['SPY'])
df_prices.info()

$DDOG: possibly delisted; no price data found  (period=4y)


Sin datos de precio para DDOG
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23899 entries, 0 to 23898
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   23899 non-null  datetime64[ns]
 1   Close   23899 non-null  float64       
 2   Ticker  23899 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 560.3+ KB


In [5]:
# Calcular los retornos mensuales, varianza del activo y covarianza con el mercado para cada ticker
df_retornos = calcular_retornos(df_prices, df_index)

# Se renombran las columnas
df_retornos = df_retornos.rename(columns={
    'Fecha' : 'Date',
    'Retorno_Mensual' : 'Monthly_Return',
    'Varianza_Activo': 'Monthly_Variance',
    'Covarianza_Mercado' : 'Market_Covariance'
})
# Quitar zonas horarias de Date y convertir a object con fecha pura
df_retornos['Date'] = df_retornos['Date'].dt.tz_localize(None).dt.date

df_retornos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19393 entries, 0 to 19392
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               19393 non-null  object 
 1   Ticker             19393 non-null  object 
 2   Monthly_Return     19391 non-null  float64
 3   Monthly_Variance   19393 non-null  float64
 4   Market_Covariance  19393 non-null  float64
dtypes: float64(3), object(2)
memory usage: 757.7+ KB


In [6]:
# Extraer datos del Balance y estado de Resultados (demora 7 minutos)
df_fundamentals = extraer_datos_fundamentales(tickers_list)
df_fundamentals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2376 entries, 0 to 2375
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Fecha                      2376 non-null   object 
 1   Total Revenue              2007 non-null   float64
 2   Gross Profit               1797 non-null   float64
 3   Operating Income           1823 non-null   float64
 4   Net Income                 1999 non-null   float64
 5   EBITDA                     1815 non-null   float64
 6   Basic Average Shares       2000 non-null   float64
 7   Cash And Cash Equivalents  2000 non-null   float64
 8   Current Debt               1592 non-null   float64
 9   Long Term Debt             1894 non-null   float64
 10  Total Debt                 1988 non-null   float64
 11  Stockholders Equity        2000 non-null   float64
 12  Total Assets               2004 non-null   float64
 13  Current Assets             1820 non-null   float

In [7]:
# Se renombran las columnas Fecha a Date
df_fundamentals = df_fundamentals.rename(columns={'Fecha': 'Date'})
df_prices = df_prices.rename(columns={'Fecha': 'Date'})

# Eliminar cualquier columna duplicada (para evitar errores al reejecutar la celda)
df_fundamentals = df_fundamentals.loc[:, ~df_fundamentals.columns.duplicated()]
df_prices = df_prices.loc[:, ~df_prices.columns.duplicated()]

# Se convierten a datetime nativo para poder hacer cálculos
df_fundamentals['Date'] = pd.to_datetime(df_fundamentals['Date']).dt.tz_localize(None)
df_prices['Date'] = pd.to_datetime(df_prices['Date']).dt.tz_localize(None)

# Cálculo del primer día del mes siguiente
df_fundamentals['Date'] = (df_fundamentals['Date'] + pd.offsets.MonthEnd(0)) + pd.Timedelta(days=1)

# Se convierten ambos DataFrames a fecha pura sin hora
df_fundamentals['Date'] = df_fundamentals['Date'].dt.date
df_prices['Date'] = df_prices['Date'].dt.date

# Asegurar que los Tickers no tengan espacios en blanco invisibles
df_prices['Ticker'] = df_prices['Ticker'].astype(str).str.strip()
df_fundamentals['Ticker'] = df_fundamentals['Ticker'].astype(str).str.strip()

# Unir fundamentals con precios mensuales
df_merged = pd.merge(
    df_prices, 
    df_fundamentals, 
    on=['Date', 'Ticker'],
    how='left'
)

# Ordenar cronológicamente para el Forward Fill
df_merged = df_merged.sort_values(by=['Ticker', 'Date'])

# Aplicar Forward Fill a las columnas financieras
cols_financieras = ['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 
                    'EBITDA', 'Basic Average Shares', 'Cash And Cash Equivalents', 'Current Debt', 
                    'Long Term Debt', 'Total Debt', 'Stockholders Equity',
                    'Total Assets', 'Current Assets', 'Current Liabilities']

df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

df_merged.info()


<class 'pandas.core.frame.DataFrame'>
Index: 23899 entries, 432 to 23898
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       23899 non-null  object 
 1   Close                      23899 non-null  float64
 2   Ticker                     23899 non-null  object 
 3   Total Revenue              19822 non-null  float64
 4   Gross Profit               17764 non-null  float64
 5   Operating Income           18025 non-null  float64
 6   Net Income                 19744 non-null  float64
 7   EBITDA                     17947 non-null  float64
 8   Basic Average Shares       19771 non-null  float64
 9   Cash And Cash Equivalents  19783 non-null  float64
 10  Current Debt               16505 non-null  float64
 11  Long Term Debt             18824 non-null  float64
 12  Total Debt                 19759 non-null  float64
 13  Stockholders Equity        19783 non-null  float6

In [8]:
# Eliminar las filas anteriores al primer reporte financiero disponible
columna_critica = 'EBITDA' # es necesaria para los ratios
df_merged_clean = df_merged.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los retornos calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_retornos, on=['Ticker', 'Date'], how='left')

df_merged_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17947 entries, 0 to 17946
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       17947 non-null  object 
 1   Close                      17947 non-null  float64
 2   Ticker                     17947 non-null  object 
 3   Total Revenue              17947 non-null  float64
 4   Gross Profit               17686 non-null  float64
 5   Operating Income           17947 non-null  float64
 6   Net Income                 17908 non-null  float64
 7   EBITDA                     17947 non-null  float64
 8   Basic Average Shares       17896 non-null  float64
 9   Cash And Cash Equivalents  17908 non-null  float64
 10  Current Debt               14980 non-null  float64
 11  Long Term Debt             17027 non-null  float64
 12  Total Debt                 17884 non-null  float64
 13  Stockholders Equity        17908 non-null  flo

In [9]:
df_with_metrics = calcular_metricas(df_merged_clean)

# Luego de calcular los ratios, se eliminan las columnas financieras absolutas
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17947 entries, 0 to 17946
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17947 non-null  object 
 1   Close               17947 non-null  float64
 2   Ticker              17947 non-null  object 
 3   Monthly_Return      17475 non-null  float64
 4   Monthly_Variance    17475 non-null  float64
 5   Market_Covariance   17475 non-null  float64
 6   MarketCap           17896 non-null  float64
 7   EnterpriseValue     17896 non-null  float64
 8   PE_Trailing         17857 non-null  float64
 9   EnterpriseToEbitda  17896 non-null  float64
 10  PriceToBook         17857 non-null  float64
 11  operatingMargins    17947 non-null  float64
 12  profitMargins       17908 non-null  float64
 13  returnOnEquity      17869 non-null  float64
 14  ReturnOnAssets      17908 non-null  float64
 15  debtToEquity        17908 non-null  float64
 16  curr

In [10]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17947 entries, 0 to 17946
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17947 non-null  object 
 1   Close               17947 non-null  float64
 2   Ticker              17947 non-null  object 
 3   Monthly_Return      17475 non-null  float64
 4   Monthly_Variance    17475 non-null  float64
 5   Market_Covariance   17475 non-null  float64
 6   MarketCap           17896 non-null  float64
 7   EnterpriseValue     17896 non-null  float64
 8   PE_Trailing         17857 non-null  float64
 9   EnterpriseToEbitda  17896 non-null  float64
 10  PriceToBook         17857 non-null  float64
 11  operatingMargins    17947 non-null  float64
 12  profitMargins       17908 non-null  float64
 13  returnOnEquity      17869 non-null  float64
 14  ReturnOnAssets      17908 non-null  float64
 15  debtToEquity        17908 non-null  float64
 16  curr

In [11]:
# Extraer datos macro (devuelve dataframe vacio si no se configuro la clave API de FRED)
indicadores = [
    'FEDFUNDS',     # Tasa de Fondos Federales (Tasa de interés de política monetaria de la Fed)
    'GS10',         # Rendimiento de los Bonos del Tesoro de EE.UU. a 10 años (Tasa libre de riesgo a largo plazo)
    'T10Y2Y',       # Diferencial de la curva de rendimientos (10 años menos 2 años, indicador de recesión)
    'CPIAUCSL',     # Índice de Precios al Consumidor (IPC / Inflación general en EE.UU.)
    'UNRATE',       # Tasa de Desempleo oficial de EE.UU.
    'AAA',          # Rendimiento de Bonos Corporativos grado de inversión con máxima calificación AAA (Moody's)
    'DTWEXAFEGS',   # Índice del Dólar Estadounidense ponderado por comercio (fortaleza del USD vs otras monedas fuertes)
    'BAMLH0A0HYM2', # Diferencial de bonos "High Yield" vs Tesoro (Termómetro de estrés de liquidez y riesgo de default corporativo)
    'INDPRO',       # Índice de Producción Industrial (Mide la economía real mensual, proxy anticipado del PIB)
    'M2SL'          # Oferta Monetaria M2 (Liquidez total en la economía, clave para entender la expansión de activos financieros)
]

df_datos_macro = extraer_datos_macro(indicadores)
df_datos_macro = df_datos_macro.sort_values(by='Fecha')
df_datos_macro.head()

,Fecha,FEDFUNDS,GS10,T10Y2Y,CPIAUCSL,UNRATE,AAA,DTWEXAFEGS,BAMLH0A0HYM2,INDPRO,M2SL
0,2022-06-30,1.21,3.14,0.06,294.957,3.6,4.24,115.4958,NaN,101.0180,21651.2
1,2022-07-31,1.68,2.90,-0.22,294.913,3.5,4.06,116.2319,NaN,101.2230,21652.8
2,2022-08-31,2.33,2.90,-0.30,295.097,3.6,4.07,118.9531,NaN,101.0963,21638.2
3,2022-09-30,2.56,3.52,-0.39,296.349,3.5,4.59,123.4836,NaN,101.2918,21536.5
4,2022-10-31,3.08,3.98,-0.41,298.007,3.6,5.10,122.7615,NaN,101.2529,21461.5


In [12]:
# Se convierten las series con crecimiento infinito a variaciones porcentuales trimestrales (QoQ) y anuales (YoY)
df_datos_macro['CPI_QoQ'] = df_datos_macro['CPIAUCSL'].pct_change(3, fill_method=None)
df_datos_macro['M2_QoQ'] = df_datos_macro['M2SL'].pct_change(3, fill_method=None)
df_datos_macro['INDPRO_QoQ'] = df_datos_macro['INDPRO'].pct_change(3, fill_method=None)

df_datos_macro['CPI_YoY'] = df_datos_macro['CPIAUCSL'].pct_change(12, fill_method=None)
df_datos_macro['M2_YoY'] = df_datos_macro['M2SL'].pct_change(12, fill_method=None)
df_datos_macro['INDPRO_YoY'] = df_datos_macro['INDPRO'].pct_change(12, fill_method=None)

# Se eliminan las columnas originales
df_datos_macro = df_datos_macro.drop(columns= ['CPIAUCSL', 'M2SL', 'INDPRO'])

# Asegurar que la columna Fecha sea tipo datetime
df_datos_macro = df_datos_macro.rename(columns = {'Fecha' : 'Date'})
df_datos_macro['Date'] = pd.to_datetime(df_datos_macro['Date'])

# 2. Desplazar la fecha al 1er día del mes, saltando un mes de publicación para evitar fuga de datos
# Ej: 2022-06-30 -> 2022-08-01
df_datos_macro['Date'] = df_datos_macro['Date'] + pd.offsets.MonthBegin(2)

# Se renombran columnas
df_datos_macro = df_datos_macro.rename(columns={
    'FEDFUNDS': 'FedFundsRate',
    'GS10': '10YTreasuryYield',
    'T10Y2Y': 'YieldCurveSpread',
    'UNRATE': 'UnemploymentRate',
    'AAA': 'AAACorporateYield',
    'DTWEXAFEGS': 'USDollarIndex',
    'BAMLH0A0HYM2': 'HighYieldSpread'
})

# Se convierte Date a tipo Object
df_datos_macro['Date'] = df_datos_macro['Date'].dt.date

In [13]:
# Se une con los datos de df_final
df_raw = pd.merge(df_final, df_datos_macro, on="Date", how= "left")
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17947 entries, 0 to 17946
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17947 non-null  object 
 1   Close               17947 non-null  float64
 2   Ticker              17947 non-null  object 
 3   Monthly_Return      17475 non-null  float64
 4   Monthly_Variance    17475 non-null  float64
 5   Market_Covariance   17475 non-null  float64
 6   MarketCap           17896 non-null  float64
 7   EnterpriseValue     17896 non-null  float64
 8   PE_Trailing         17857 non-null  float64
 9   EnterpriseToEbitda  17896 non-null  float64
 10  PriceToBook         17857 non-null  float64
 11  operatingMargins    17947 non-null  float64
 12  profitMargins       17908 non-null  float64
 13  returnOnEquity      17869 non-null  float64
 14  ReturnOnAssets      17908 non-null  float64
 15  debtToEquity        17908 non-null  float64
 16  curr

In [14]:
# Guardar datos extraidos en fichero raw_data
df_raw.to_parquet(f"{data_folder}/raw_data.parquet")